# Промпт #14 — Structured output JSON edge case

**Техника:** Структурированный вывод в JSON  
**Задача:** Проверить что будет если данных в тексте не хватает  
**Сложность:** ⭐⭐⭐☆☆

In [1]:
import sys
import json
sys.path.append('..')
from config import get_completion

In [2]:
# Вакансия где намеренно нет некоторых данных
text = """Ищем крутого разработчика в стартап. 
Работа интересная, команда дружная. 
Пиши на почту jobs@startup.ru"""

prompt = f"""Извлеки данные из вакансии и верни строго в JSON формате.
Без дополнительного текста, только JSON.
Если данных нет — ставь null.

<vacancy>{text}</vacancy>

Формат:
{{
    "должность": "",
    "компания": "",
    "зарплата": "",
    "опыт": "",
    "локация": "",
    "требования": []
}}"""

result = get_completion(prompt)
print(result)

try:
    parsed = json.loads(result)
    print("\n✅ Валидный JSON!")
    print(parsed)
except:
    print("\n❌ Невалидный JSON")

{
    "должность": "разработчик",
    "компания": "стартап",
    "зарплата": null,
    "опыт": null,
    "локация": null,
    "требования": []
}

✅ Валидный JSON!
{'должность': 'разработчик', 'компания': 'стартап', 'зарплата': None, 'опыт': None, 'локация': None, 'требования': []}


## Оценка: 4/5

## Инсайт
Инструкция "если данных нет — ставь null" частично 
защищает от галлюцинаций в JSON формате.

**Сравнение с #08 (XML edge case):**
- XML: пустые теги когда данных нет
- JSON: null когда данных нет
- Оба варианта не выдумывают данные ✅

**Что модель сделала правильно:**
- "стартап" → компания (логичный вывод)
- "разработчик" → должность (из контекста)
- зарплата/опыт/локация → null (данных нет) ✅

**Что модель сделала неточно:**
- требования → [] вместо null ❌
- данных о требованиях нет, но модель 
  поставила пустой массив а не null

**Вывод:** для массивов нужна отдельная инструкция —
"если список пустой — ставь null, не []"
Общей инструкции про null недостаточно.